# Практика 1. OpenRouter и выбор модели

**Курс:** Мультиагентные системы · **Дата:** 02.10.2026

**Цель:** научиться делать запросы к ЛЛМ через API и осознанно выбрать свою модель на семестр — она станет основой вашей курсовой.

**План:**
1. Первый запрос к модели
2. Своя задача — три модели
3. Измерения: latency и длина ответа
4. Сводная таблица по критериям
5. Финальный выбор + обоснование
6. Бонус для опытных: стриминг и temperature

**Что понадобится:** API-ключ с openrouter.ai (регистрация — VPN может понадобиться), Python 3, библиотека `requests`.

## Подготовка

Ключ вставляем через `getpass` — он не сохранится в ноутбуке, и вы не случайно не выложите его в гит.

In [1]:
# Если requests ещё не установлен — раскомментируйте:
# !pip install requests

import requests
from getpass import getpass

API_KEY = getpass("Введите свой API ключ: ")
URL = "https://openrouter.ai/api/v1/chat/completions"

## Задание 1. Первый запрос

Функция `ask` — минимальный клиент чата: собирает сообщения, отправляет на OpenRouter, возвращает текст ответа. Разберитесь, что здесь происходит — на следующих практиках будем наращивать именно эту функцию.

In [2]:
def ask(model: str, prompt: str, system: str = None) -> str:
    """Один запрос к модели через OpenRouter. Возвращает текст ответа."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": model, "messages": messages},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

# Запускаем: модель — лидер голосования группы на ДЗ-1
print(ask("nvidia/nemotron-3-ultra-550b-a55b:free", "Привет! Представься в одном предложении."))

HTTPError: 403 Client Error: Forbidden for url: https://openrouter.ai/api/v1/chat/completions

Если получили ошибку — проверьте:
- `401` — неверный ключ;
- `404` / `модели нет` — бесплатный парк волатилен, загляните на https://openrouter.ai/models?max_price=0 и возьмите живую модель с суффиксом `:free`;
- `429` — упёрлись в лимит (примерно 20 запросов в минуту), подождите минуту.

## Задание 2. Своя задача — три модели

Впишите **свою** задачу — из учебы, работы или pet-проекта. Она поедет с вами до конца семестра, так что выбирайте ту, что реально интересна. Хорошая задача: конкретная, проверяемая глазами, не слишком простая.

In [ ]:
TASK = "Я хочу создать Telegram канал, посвященный одежде из Китая. У меня есть папка в Discord, в которой различные производители публикуют свои обновления и релизы. Мне необходимо реализовать инструмент, который будет способен отслеживать изменения у производителей, предлагать посты с новыми релизами, и в случае ручного одобрения публиковать новость в телеграмм канал."  # <-- впишите сюда свою задачу

assert TASK, "Сначала впишите задачу!"

MODELS = [
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "qwen/qwen3.8-27b:free",
    "poolside/laguna-s-2.1:free",
    # добавьте или замените по вкусу: https://openrouter.ai/models?max_price=0
]


for model in MODELS:
    print(f"\n{'=' * 60}\n{model}\n{'=' * 60}")

    try:
        print(ask(model, TASK))
    except requests.exceptions.HTTPError as e:
        print(f"Ошибка HTTP: {e}")
        print("Ответ сервера:", e.response.text if e.response is not None else "нет данных")
    except requests.exceptions.RequestException as e:
        print(f"Ошибка запроса: {e}")


nvidia/nemotron-3-ultra-550b-a55b:free
Это отличная задача для автоматизации контент-маркетинга. По сути, вам нужен **конвейер (pipeline): Discord Listener → Processor/AI Enricher → Admin Panel (Moderation Queue) → Telegram Publisher**.

Ниже подробная архитектура, стек технологий, пошаговый план реализации и примеры кода для MVP.

---

### 1. Архитектура решения

```mermaid
graph LR
    A[Discord Server<br/>(Manufacturers Channels)] -->|Webhook / Bot API| B(Python Worker:<br/>Listener & Parser)
    B --> C{Filter & Dedup:<br/>New release?}
    C -- Yes --> D[AI Enricher:<br/>Generate Post Text,<br/>Translate, Select Images]
    D --> E[(Database:<br/>PostgreSQL/SQLite<br/>Status: PENDING)]
    E --> F[Admin Panel:<br/>Web UI or Telegram Bot<br/>(Approve/Reject/Edit)]
    F -- Approve --> G[Publisher Worker]
    G --> H[Telegram Channel]
    F -- Reject/Edit --> E
```

---

### 2. Рекомендуемый стек технологий

| Компонент | Технология | Почему |
| :--- | :--- | :--- |
| **Язык** | **

Прочитайте три ответа. Что различается: точность? стиль? уверенность? Где модель «льёт воду»? Запишите 2–3 наблюдения — они пригодятся в задании 5.

## Задание 3. Измеряем: latency и длина

Субъективное «мне понравился ответ» — это хорошо, но инженерия начинается с измерений.

In [ ]:
import time

results = []
for model in MODELS:
    t0 = time.perf_counter()
    answer = ask(model, TASK)
    dt = time.perf_counter() - t0
    results.append({"model": model, "latency": dt, "chars": len(answer)})
    print(f"{model:45s} {dt:6.1f} c   {len(answer):6d} символов")

# Вопросы к себе:
# - коррелирует ли «дольше» с «лучше»?
# - одинаковый ли смысл ответов, или кто-то ответил на другую задачу?
# - сколько примерно токенов в ответе (грубая оценка: 1 токен ~ 4 символа)?

## Задание 4. Сводная таблица по критериям

Критерии — с лекции. Данные берите со страницы модели на OpenRouter и с https://artificialanalysis.ai. Заполните таблицу для своих кандидатов:

| Модель | Контекст (тыс. ток.) | Intelligence Index | Latency | Политика данных | Открытые веса |
|---|---|---|---|---|---|
| ... | | | | | |

**Обязательно** проверьте политику данных: учится ли провайдер на ваших запросах на бесплатном тарифе. Если в вашей задаче есть рабочий код или персональные данные — это критерий №1.

## Задание 5. Моя основная модель на семестр

Заполните и оставьте в ноутбуке — это зачаток вашей курсовой:

**Основная модель:** \...
**Почему именно она** (2–3 предложения, со ссылкой на измерения из заданий 3–4): ...

**Что я буду делать с ней дальше** (идея для курсовой — сравнение промптов, агент с инструментами, измерение деградации контекста, ...): ...

**Лог миграций** (заполняется по ходу семестра, если смените модель):

| Дата | Было | Стало | Почему |
|---|---|---|---|
| | | | |

Менять модель можно в любой момент — но с обоснованием: все практики и курсовая считаются на основной модели, чтобы ваши измерения были честными (улучшили промпт — а не просто сменили модель). Это как с зависимостями в проекте: фиксируешь версию для воспроизводимости, обновляешь осознанно.

## Бонус (для опытных)

### Бонус 1. Стриминг

Ответ приходит по кускам, как в чате, — не ждём весь текст:

In [ ]:
import json

def ask_stream(model: str, prompt: str):
    with requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": model, "messages": [{"role": "user", "content": prompt}], "stream": True},
        stream=True, timeout=120,
    ) as r:
        for line in r.iter_lines():
            if line and line.startswith(b"data: ") and line != b"data: [DONE]":
                chunk = json.loads(line[6:])
                delta = chunk["choices"][0]["delta"].get("content", "")
                print(delta, end="", flush=True)
    print()

ask_stream(MODELS[0], "Расскажи короткий анекдот про программиста.")

### Бонус 2. Temperature

«Температура» — ручка случайности: 0 — почти детерминизм, выше — творческий хаос. Сравните:

In [ ]:
for temp in [0.0, 0.7, 1.5]:
    resp = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": MODELS[0],
            "messages": [{"role": "user", "content": "Придумай название для рок-группы"}],
            "temperature": temp,
        },
        timeout=120,
    ).json()
    print(f"temperature={temp}: {resp['choices'][0]['message']['content']}")